# Getting Grouper results out of Python

Grouper enumerates molecular structures as `GroupGraph` objects. For those structures to be useful in molecular dynamics (GROMACS, OpenMM, LAMMPS), docking (AutoDock Vina, Glide), quantum chemistry (Gaussian, ORCA, Psi4), database lookup (PubChem, ChEMBL, NIST), or downstream Python ML pipelines (pandas, JSONL, HuggingFace datasets), you need them out of memory and into a standard format.

This notebook covers two halves of that story:

**Per-molecule format export** — methods on a single `GroupGraph` that produce file-format strings:

| method | format | typical consumer |
|---|---|---|
| `gG.to_3d()` | RDKit Mol with 3D coords | further RDKit / Python work |
| `gG.to_sdf()` | SDF (single-mol) | KNIME, Pipeline Pilot, RDKit pipelines |
| `gG.to_mol()` | V2000 MOL block | legacy single-molecule tools |
| `gG.to_xyz()` | XYZ coords | Gaussian, ORCA, NWChem, Psi4 |
| `gG.to_pdb()` | PDB | PyMOL, VMD, AutoDock Vina |
| `gG.to_inchi()` / `gG.to_inchi_key()` | InChI / InChIKey | PubChem, ChEMBL, NIST WebBook lookup |
| `gG.to_smarts()` | SMARTS pattern | substructure querying |
| `gG.visualize()` | matplotlib port-graph plot | inline notebook diagrams |

**Batch export over a `GroupGraphSet`** — `exhaustive_generate`, `random_generate`, and `exhaustive_fragment` all return a `GroupGraphSet`:

```python
from Grouper import exhaustive_generate, random_generate, exhaustive_fragment
results = exhaustive_generate(n, node_defs)        # enumerate every graph at size n
results = random_generate(n, node_defs, k, seed)   # k random samples
results = exhaustive_fragment(smiles, node_defs)   # all decompositions of a molecule
```

The `GroupGraphSet` then exposes one-shot batch methods for the entire library:

| method | output | use |
|---|---|---|
| `results.to_dataframe()` | pandas DataFrame | screening / analysis in Python |
| `results.to_csv(path)` | CSV file | spreadsheets, KNIME, Excel |
| `results.to_jsonl(path)` | JSONL file | ML datasets (HuggingFace, JAX) |
| `results.to_sdf(path)` | multi-mol SDF | downstream chemistry tools |
| `results.to_smiles_list()` | `list[str]` | quick canonical-SMILES list |
| `results.filter(predicate)` | new `GroupGraphSet` | subset by user-supplied condition |
| `results.sample(n, seed=)` | new `GroupGraphSet` | random subset |

Both halves accept optional property estimators (e.g. Joback, on a sibling branch) so predicted thermophysical properties can be attached to every record without a separate join step.

**Run all cells** to see outputs. Estimated runtime: under 10 seconds.

Requirements: `Grouper`, `rdkit`, `pandas`. Optional for the in-notebook 3D view: `py3Dmol`.

In [1]:
import tempfile
import os
from pathlib import Path

from rdkit import Chem
from rdkit.Chem import Draw

from Grouper import Group, GroupGraph, exhaustive_generate
from Grouper.exports import to_3d_mol, to_sdf, EmbedError

## 1. Build a GroupGraph the usual way

Construct n-propanol (CCCO) by hand from a 3-carbon `alkyl` group plus a `hydroxyl` — small enough that every `to_xyz`/`to_pdb`/`to_sdf` cell below prints a scannable block, and concrete enough to show that a single `Group` can encapsulate a multi-atom fragment (this is the whole point of Grouper over working with RDKit atom-by-atom).

In real usage you'd skip the manual construction and get a `GroupGraph` from one of:

```python
from Grouper import exhaustive_generate, random_generate, fragment
gG  = next(iter(exhaustive_generate(n, node_defs)))   # one of many enumerated
gG  = next(iter(random_generate(n, node_defs, k, seed)))
gGs = fragment(smiles, node_defs)                     # list of decompositions
```

In [ ]:
gG = GroupGraph()
gG.add_node("hydroxyl", "O", [0])  # 1 port
gG.add_node("alkyl", "CCC", [0])  # 1 port on the leftmost C of CCC
gG.add_edge((0, 0), (1, 0))  # hydroxyl — alkyl

print(f"SMILES:  {gG.to_smiles()}")
print(f"Nodes:   {len(gG.nodes)} groups")
Draw.MolToImage(Chem.MolFromSmiles(gG.to_smiles()), size=(300, 200))

## 2. Generate 3D coordinates

`gG.to_3d()` runs RDKit's ETKDG distance-geometry embedding followed by force-field minimization. Default force field is **MMFF94** (more accurate); falls back to **UFF** automatically if MMFF lacks parameters for some atom in your molecule. Pass `force_field="uff"` to force UFF, or `"none"` to skip minimization (just the embed).

In [3]:
mol_3d = gG.to_3d(seed=42)
print(f"Atoms (incl. explicit H):  {mol_3d.GetNumAtoms()}")
print(f"Conformers:                 {mol_3d.GetNumConformers()}")

conf = mol_3d.GetConformer()
print("\nFirst 5 atom positions (Å):")
print(f"  {'idx':>3}  {'symbol':>6}  {'x':>8}  {'y':>8}  {'z':>8}")
for i in range(min(5, mol_3d.GetNumAtoms())):
    a = mol_3d.GetAtomWithIdx(i)
    p = conf.GetAtomPosition(i)
    print(f"  {i:>3}  {a.GetSymbol():>6}  {p.x:>8.3f}  {p.y:>8.3f}  {p.z:>8.3f}")

Atoms (incl. explicit H):  12
Conformers:                 1

First 5 atom positions (Å):
  idx  symbol         x         y         z
    0       C    -1.306    -0.015     0.502
    1       C    -0.141     0.649    -0.217
    2       C     0.913    -0.356    -0.663
    3       O     1.492    -1.013     0.456
    4       H    -2.054     0.734     0.782


### Optional: 3D view in the notebook

If `py3Dmol` is installed, render the conformer interactively. Skip this cell if not.

In [4]:
try:
    import py3Dmol

    view = py3Dmol.view(width=300, height=300)
    view.addModel(Chem.MolToMolBlock(mol_3d), "mol")
    view.setStyle({"stick": {}, "sphere": {"scale": 0.25}})
    view.zoomTo()
    view.show()
except ImportError:
    print(
        "(py3Dmol not installed — skipping interactive view; install via `pip install py3Dmol`)"
    )

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## 3. The four file formats

Each export method returns a string when called with no arguments — useful for embedding in a notebook or piping to another tool. Pass `path=...` to write to disk instead (returns `None`).

In [5]:
for fmt in ("sdf", "mol", "xyz", "pdb"):
    text = getattr(gG, f"to_{fmt}")()
    print(f"=== gG.to_{fmt}() ===")
    # Show first 8 lines + total length so we don't drown the cell
    for line in text.splitlines()[:8]:
        print(f"  {line}")
    print(f"  ... ({len(text.splitlines())} lines total)\n")

=== gG.to_sdf() ===
  
       RDKit          3D
  
   12 11  0  0  0  0  0  0  0  0999 V2000
     -1.3061   -0.0154    0.5024 C   0  0  0  0  0  0  0  0  0  0  0  0
     -0.1415    0.6492   -0.2174 C   0  0  0  0  0  0  0  0  0  0  0  0
      0.9134   -0.3560   -0.6626 C   0  0  0  0  0  0  0  0  0  0  0  0
      1.4916   -1.0131    0.4565 O   0  0  0  0  0  0  0  0  0  0  0  0
  ... (29 lines total)

=== gG.to_mol() ===
  
       RDKit          3D
  
   12 11  0  0  0  0  0  0  0  0999 V2000
     -1.3061   -0.0154    0.5024 C   0  0  0  0  0  0  0  0  0  0  0  0
     -0.1415    0.6492   -0.2174 C   0  0  0  0  0  0  0  0  0  0  0  0
      0.9134   -0.3560   -0.6626 C   0  0  0  0  0  0  0  0  0  0  0  0
      1.4916   -1.0131    0.4565 O   0  0  0  0  0  0  0  0  0  0  0  0
  ... (28 lines total)

=== gG.to_xyz() ===
  12
  
  C     -1.306081   -0.015390    0.502396
  C     -0.141451    0.649158   -0.217435
  C      0.913381   -0.356018   -0.662629
  O      1.491570   -1.013051    0.4

## 4. Writing to disk

Same methods, but with a `path` argument. Returns `None`; the file appears at the given path.

In [6]:
with tempfile.TemporaryDirectory() as td:
    out_dir = Path(td)
    for fmt in ("sdf", "mol", "xyz", "pdb"):
        path = out_dir / f"propanol.{fmt}"
        getattr(gG, f"to_{fmt}")(path=str(path))
        print(f"  wrote {path.name}: {path.stat().st_size:>5} bytes")

  wrote propanol.sdf:  1060 bytes
  wrote propanol.mol:  1055 bytes
  wrote propanol.xyz:   484 bytes
  wrote propanol.pdb:  1079 bytes


## 5. Attaching properties to an SDF

SDF supports arbitrary `> <KEY>` property blocks per molecule. Useful for shipping predictions, source metadata, or experimental measurements alongside the structure so downstream tools can use them without a separate join step.

Pass `properties=...` (a dict) to `to_sdf` for a single molecule:

In [ ]:
sdf = gG.to_sdf(
    name="propanol",
    properties={
        "smiles": gG.to_smiles(),
        "source": "Grouper.exhaustive_generate",
        "library": "{alkyl, hydroxyl}",
        "comment": "hand-built example",
    },
)
# Show only the property-block section (after `M  END`)
print(sdf[sdf.find("M  END") :])

## 6. Identifier formats: InChI, InChIKey, SMARTS

For database lookup (PubChem, ChEMBL, NIST WebBook) and substructure querying, you need 1D string identifiers — not coordinates. Three methods give you the canonical chemistry identifiers:

- `gG.to_inchi()` — the IUPAC InChI, layered and deterministic across implementations (canonical SMILES isn't, by contrast).
- `gG.to_inchi_key()` — the 27-character hash form. Fixed-length, URL-safe, the primary key form most chemistry databases use.
- `gG.to_smarts()` — SMARTS pattern, useful when a generated structure becomes a query for downstream substructure search.

In [8]:
print(f"InChI:     {gG.to_inchi()}")
print(
    f"InChIKey:  {gG.to_inchi_key()}     # 27 chars, primary key for PubChem/ChEMBL/NIST"
)
print(f"SMARTS:    {gG.to_smarts()}")

InChI:     InChI=1S/C3H8O/c1-2-3-4/h4H,2-3H2,1H3
InChIKey:  BDERNNFJNOPAEC-UHFFFAOYSA-N     # 27 chars, primary key for PubChem/ChEMBL/NIST
SMARTS:    [#6]-[#6]-[#6]-[#8]


## 7. Batch SDF: an entire `exhaustive_generate` run with metadata

The realistic industrial use case. Generate a small library, pass a `properties=` callback that runs per-molecule, and write everything into a multi-molecule SDF that downstream pipelines can stream through.

`skip_failures=True` keeps the batch going past molecules whose 3D embedding fails (rare but not impossible — strained ring systems can defeat ETKDG). Each failure emits a `UserWarning` with the offending SMILES.

In [9]:
# Enumerate every n=3 GroupGraph over a {methyl, hydroxyl} library.
node_defs = {
    Group("methyl", "C", [0, 0, 0, 0]),
    Group("hydroxyl", "O", [0, 0]),
}
results = exhaustive_generate(3, node_defs, num_procs=1)
print(f"exhaustive_generate(n=3, |library|=2) -> {len(results)} unique molecules")


# Define a per-molecule properties callback. In a real screening flow
# this would call JobackEstimate / a docking score / an ML predictor.
def attach_metadata(g):
    smi = g.to_smiles()
    return {
        "smiles": smi,
        "n_nodes": len(g.nodes),
        "has_OH": "yes" if "O" in smi else "no",
    }


with tempfile.TemporaryDirectory() as td:
    out = Path(td) / "library.sdf"
    n = to_sdf(results, path=str(out), properties=attach_metadata, skip_failures=True)
    print(
        f"\nwrote {n} of {len(results)} molecules to {out.name} ({out.stat().st_size} bytes)"
    )

    # Round-trip via RDKit's SDMolSupplier and pull out the property tags.
    supplier = Chem.SDMolSupplier(str(out))
    print("\nRound-tripped property block for each molecule in the SDF:")
    print(f"  {'SMILES':<8}  {'has_OH':<8}  {'n_nodes':<8}")
    for mol in supplier:
        if mol is None:
            continue
        print(
            f"  {mol.GetProp('smiles'):<8}  {mol.GetProp('has_OH'):<8}  {mol.GetProp('n_nodes'):<8}"
        )

Using 1 processors (streaming)
100% (10/10)
Number of unique graphs: 10
exhaustive_generate(n=3, |library|=2) -> 10 unique molecules

wrote 10 of 10 molecules to library.sdf (7407 bytes)

Round-tripped property block for each molecule in the SDF:
  SMILES    has_OH    n_nodes 
  OCO       yes       3       
  C1OO1     yes       3       
  CCC       no        3       
  OOO       yes       3       
  CCO       yes       3       
  C1CO1     yes       3       
  O1OO1     yes       3       
  C1CC1     no        3       
  COO       yes       3       
  COC       yes       3       


>A geng -cd1D2 n=3 e=2-3
>A vcolg -m2T
>Z 2 graphs generated in 0.00 sec
>Z 2 graphs read from stdin; 10 coloured graphs written to stdout; 0.00 sec


## 8. `GroupGraphSet`: batch operations on a whole result

`exhaustive_generate`, `random_generate`, and `exhaustive_fragment` return a `GroupGraphSet` — a `set[GroupGraph]` subclass with batch-conversion methods bolted on. It IS a `set` (iteration, `len`, `in`, `isinstance(x, set)` all work), so existing code that treats results as plain sets keeps working unchanged.

The methods take care of the boilerplate every screening notebook used to write by hand: flattening to a DataFrame, picking a random subset, filtering by predicate, writing to CSV / JSONL / SDF. They all accept the same `properties=[...]` argument, so any predicted property (Joback, ML, custom) can be attached to every output format with one call.

`results` from this point on is the same `exhaustive_generate(3, {methyl, hydroxyl})` library used in the previous section.

In [10]:
print(f"type(results) = {type(results).__name__}")
print(
    f"isinstance(results, set) = {isinstance(results, set)}    # legacy code keeps working"
)
print(f"len(results) = {len(results)}")
print(f"\nrepr(results):\n  {results}")

# to_smiles_list — fastest path to a Python list of canonical SMILES.
print("\nresults.to_smiles_list() (sorted for display):")
print(f"  {sorted(results.to_smiles_list())}")

# to_dataframe — adds smiles + n_nodes columns by default. Pass
# properties=[...] to attach predicted property columns.
print('\nresults.to_dataframe(sort_by="smiles"):')
df = results.to_dataframe(sort_by="smiles")
print(df.to_string())

type(results) = GroupGraphSet
isinstance(results, set) = True    # legacy code keeps working
len(results) = 10

repr(results):
  GroupGraphSet(10 graphs: ['OCO', 'C1OO1', 'CCC', 'OOO', 'CCO'], ... (5 more))

results.to_smiles_list() (sorted for display):
  ['C1CC1', 'C1CO1', 'C1OO1', 'CCC', 'CCO', 'COC', 'COO', 'O1OO1', 'OCO', 'OOO']

results.to_dataframe(sort_by="smiles"):


  smiles  n_nodes
0  C1CC1        3
1  C1CO1        3
2  C1OO1        3
3    CCC        3
4    CCO        3
5    COC        3
6    COO        3
7  O1OO1        3
8    OCO        3
9    OOO        3


In [11]:
from pathlib import Path

# Filter — chained calls keep the GroupGraphSet type, so .to_dataframe()
# still works after a .filter().
print("results.filter(lambda g: 'O' in g.to_smiles()):")
o_only = results.filter(lambda g: "O" in g.to_smiles())
print(f"  {sorted(o_only.to_smiles_list())}")

# Sample — reproducible with a seed.
print("\nresults.sample(3, seed=42).to_smiles_list():")
print(f"  {sorted(results.sample(3, seed=42).to_smiles_list())}")

# CSV / JSONL — defaults to sort_by="smiles" for byte-deterministic
# output across runs.
with tempfile.TemporaryDirectory() as td:
    csv_path = Path(td) / "library.csv"
    jsonl_path = Path(td) / "library.jsonl"
    results.to_csv(str(csv_path))
    results.to_jsonl(str(jsonl_path))
    print(f"\nresults.to_csv  -> {csv_path.stat().st_size} bytes")
    print(f"results.to_jsonl -> {jsonl_path.stat().st_size} bytes")
    print("\nFirst 3 lines of the CSV:")
    with open(csv_path) as f:
        for line in f.readlines()[:3]:
            print(f"  {line.rstrip()}")
    print("\nFirst 3 lines of the JSONL:")
    with open(jsonl_path) as f:
        for line in f.readlines()[:3]:
            print(f"  {line.rstrip()}")

results.filter(lambda g: 'O' in g.to_smiles()):
  ['C1CO1', 'C1OO1', 'CCO', 'COC', 'COO', 'O1OO1', 'OCO', 'OOO']

results.sample(3, seed=42).to_smiles_list():
  ['C1OO1', 'CCO', 'OCO']

results.to_csv  -> 83 bytes
results.to_jsonl -> 298 bytes

First 3 lines of the CSV:
  smiles,n_nodes
  C1CC1,3
  C1CO1,3

First 3 lines of the JSONL:
  {"smiles":"C1CC1","n_nodes":3}
  {"smiles":"C1CO1","n_nodes":3}
  {"smiles":"C1OO1","n_nodes":3}


## 9. Force-field choices and the embed pipeline

RDKit's ETKDG distance-geometry embedding gives a reasonable starting structure; the force-field minimization step relaxes it to a local energy minimum. For most organic molecules in Joback's element scope, MMFF94 (default) and UFF give similar results. UFF covers more atom types, MMFF is generally more accurate where parameters exist.

In [ ]:
for ff in ("mmff94", "uff", "none"):
    m = gG.to_3d(seed=42, force_field=ff)
    conf = m.GetConformer()
    # End-to-end distance proxy: oxygen to the carbon farthest from it
    # (the terminal methyl). Robust to whatever heavy-atom index order
    # to_atom_graph hands back.
    ox_idx = next(a.GetIdx() for a in m.GetAtoms() if a.GetSymbol() == "O")
    p_o = conf.GetAtomPosition(ox_idx)

    def _dist_to_o(ci):
        p = conf.GetAtomPosition(ci)
        return ((p.x - p_o.x) ** 2 + (p.y - p_o.y) ** 2 + (p.z - p_o.z) ** 2) ** 0.5

    c_indices = [a.GetIdx() for a in m.GetAtoms() if a.GetSymbol() == "C"]
    far_c = max(c_indices, key=_dist_to_o)
    d = _dist_to_o(far_c)
    print(f"  force_field={ff!r:>10}  C–O end-to-end distance: {d:.3f} Å")

## 10. When embedding fails

ETKDG isn't infallible — strained polycyclics, unusual valences, or just bad luck with the random seed can leave it without a feasible solution. The export module raises `EmbedError` with the offending SMILES, so a caller can decide whether to retry, skip, or fail the batch.

In [13]:
import warnings

# Garbage SMILES is the easiest way to trigger the error path.
try:
    to_3d_mol("not a smiles")
except EmbedError as e:
    print("Caught EmbedError:")
    print(f"  {e}")

# In a batch, you usually want to skip the failure rather than abort.
# to_sdf(skip_failures=True) does this for you and emits a warning.
print("\nBatch with skip_failures=True:")
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    with tempfile.NamedTemporaryFile(mode="w", suffix=".sdf", delete=False) as tmp:
        out_path = tmp.name
    n = to_sdf(
        ["CCO", "not a smiles", "CC"],
        out_path,
        skip_failures=True,
    )
    print(f"  wrote {n} of 3 molecules")
    for w in caught:
        print(f"  warning: {w.message}")
    os.unlink(out_path)

Caught EmbedError:
  could not parse SMILES: 'not a smiles'

Batch with skip_failures=True:
  wrote 2 of 3 molecules


[21:26:09] SMILES Parse Error: syntax error while parsing: not
[21:26:09] SMILES Parse Error: check for mistakes around position 3:
[21:26:09] not
[21:26:09] ~~^
[21:26:09] SMILES Parse Error: Failed parsing SMILES 'not' for input: 'not'
[21:26:09] SMILES Parse Error: syntax error while parsing: not
[21:26:09] SMILES Parse Error: check for mistakes around position 3:
[21:26:09] not
[21:26:09] ~~^
[21:26:09] SMILES Parse Error: Failed parsing SMILES 'not' for input: 'not'


## 11. Composing with other Grouper features

Two patterns worth noting:

**Predicted properties → SDF / DataFrame**: if you've installed the property-estimation branch, attach Joback predictions directly:

```python
from Grouper.properties import JobackEstimate

# As SDF tags
to_sdf(
    results, "out.sdf",
    properties=lambda g: dict(JobackEstimate.from_group_graph(g)),
)

# Or as a pandas DataFrame
df = results.to_dataframe(properties=["joback"])
```

Each molecule then carries `joback.Tb`, `joback.Tc`, `joback.Pc`, `joback.Vc`, `joback.MW`, etc. as columns or SD property tags.

**Round-trip via SMILES**: every export function accepts raw SMILES strings, not just `GroupGraph`s, so they double as a SMILES-conversion utility:

```python
from Grouper.exports import to_3d_mol, to_pdb, to_inchi_key
to_pdb("CCO", path="ethanol.pdb")               # SMILES -> 3D PDB on disk
to_inchi_key("CC(=O)Oc1ccccc1C(=O)O")           # aspirin's InChIKey for PubChem lookup
```

## 12. Summary

| You have... | Call... | You get... |
|---|---|---|
| a `GroupGraph` from any source | `gG.to_sdf()` | SDF string (or write to file) |
| a `GroupGraph`, want a database key | `gG.to_inchi_key()` | 27-char hash for PubChem/ChEMBL |
| a `GroupGraphSet` from `exhaustive_generate` | `results.to_dataframe()` | a pandas table |
| a `GroupGraphSet`, need a screening file | `results.to_csv("out.csv")` or `.to_jsonl()` | analysis-ready file |
| a `GroupGraphSet` + property predictor | `results.to_sdf("out.sdf", properties=["joback"])` | one multi-mol SDF with predictions |
| a `GroupGraphSet`, need a subset to inspect | `results.sample(10).to_dataframe()` | 10 random molecules in a table |
| a SMILES string | `to_pdb(smiles, path=...)` | 3D PDB |

The export module turns Grouper from "a Python tool for enumerating molecules" into a building block of a larger pipeline that ends in MD, docking, quantum chemistry, database lookup, or ML training.